In [23]:
!pip install transformers
!pip install accelerate
!pip install sentence-transformers
!pip install faiss-cpu
!pip install PyMuPDF
!pip install langchain-community

In [24]:
import fitz  # PyMuPDF
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def load_and_split_pdf_with_source(pdf_path):
    loader = PyMuPDFLoader(pdf_path)
    documents = loader.load()
    for doc in documents:
        doc.metadata["source"] = "term_paper"
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    return splitter.split_documents(documents)

pdf_path = "/content/cse316 final termpaper.pdf"
chunks = load_and_split_pdf_with_source(pdf_path)


In [25]:
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

def create_vector_store(chunks):
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(chunks, embedding=embeddings)
    return vector_store

vector_store = create_vector_store(chunks)


In [30]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load lightweight Falcon model
tokenizer = AutoTokenizer.from_pretrained("tiiuae/falcon-rw-1b")
model = AutoModelForCausalLM.from_pretrained("tiiuae/falcon-rw-1b")

def ask_question_falcon(vector_store, question):
    # Search relevant documents from vector store
    docs = vector_store.similarity_search(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    # Create prompt with context
    prompt = f"Answer the following question based on the context below:\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the generated answer (remove the prompt part)
    if "Answer:" in answer:
        return answer.split("Answer:")[-1].strip()
    return answer.strip()


tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

In [32]:
question = "How does the paper classify file systems?"
answer = ask_question_falcon(vector_store, question)
print("🧠 Answer:", answer)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🧠 Answer: The paper classifies file systems into three categories: 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
•


What is the main objective of this paper? (You already tried this)

What topics are covered in this paper?

What is a file system according to the document?

Why is the file system important in computing?



What types of file systems are discussed in the paper?

What are the main components of a file system?

How does the paper classify file systems?

What examples of file allocation methods are mentioned?



How does FAT file system differ from NTFS in this paper?

What are the advantages and disadvantages of hierarchical file structures?

Which file system does the paper suggest is most efficient and why?



Where are file systems commonly used in real-world computing?

What role does the file system play in operating systems like Linux or Windows?



What is the final conclusion of the paper?

What future work or directions are suggested in the paper?